# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

md = dataset.metadata
print(f"Dataset title: {md.name}")
print(f"Description: {md.description}")
print(f"Published: {md.datePublished}")
print(f"Identifier: {md.identifier}")
print(f"Fields: {md.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields
print("Available record sets and their fields:")
for record_set in md.record_sets:
    print(f"- Record Set: {record_set['@id']} (Name: {record_set.get('name', '<unnamed>')})")
    if 'field' in record_set:
        fields = record_set['field'] if isinstance(record_set['field'], list) else [record_set['field']]
        for field in fields:
            print(f"    - Field: {field['@id']} (Name: {field.get('name', '<unnamed>')}, Data Type: {field.get('dataType', '<unknown>')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Extract data from record sets
# First, get all available record set @ids
record_set_ids = [record_set['@id'] for record_set in md.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

print("Loaded DataFrames:")
for rsid in dataframes:
    print(f"- {rsid}: {list(dataframes[rsid].columns)}")

# Select first record set as example for further analysis
example_rs_id = record_set_ids[0]
print(f"\nPreview of DataFrame for record set {example_rs_id} (first 5 rows):")
display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field from the example record set.
# For demonstration, we'll try to select the first numeric column if available.

df = dataframes[example_rs_id]
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Attempt to convert candidates to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue

if numeric_field is not None:
    print(f"Numeric field selected: {numeric_field}")
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} greater than mean ({threshold}):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field if available
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable non-numeric field found for grouping.")
else:
    print("No numeric field could be detected in this record set. Please inspect the columns for numeric analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in record set {example_rs_id}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides structured records and associated metadata suitable for exploratory data analysis using the `mlcroissant` library.
- Using Croissant `@id` references ensures precise and reproducible data operations across large scientific datasets.
- The outlined steps (loading, overview, extraction, EDA, and visualization) can be repeated for other record sets and fields in the dataset using their `@id`s for deep, domain-specific explorations.

*For more information, visit the [FAIR² metadata page](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).*